In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import string
import os

# Force file paths to be relative to the script location (works in both Python script & Jupyter)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

INPUT_TEXT_FILE = os.path.join(BASE_DIR, 'The home book of verse processed.txt')

# Default Enigma configuration
DEFAULT_ROTOR_ORDER = ['III', 'II', 'I']  # Left-Middle-Right
DEFAULT_POSITIONS = [0, 0, 0]  # Starting positions (A-A-A)

# Training configuration
MESSAGE_LENGTH = 100  # How many characters to use per training sample
NUM_TRAINING_SAMPLES = 5000  # Number of encrypted samples to generate

# ============================================================================
# ENIGMA SIMULATOR (Simplified - no plugboard)
# ============================================================================
class SimpleEnigma:
    """Simplified Enigma machine simulator"""
    
    # Historical rotor wirings (I-V)
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK'
    }
    
    # Notch positions (when rotor steps the next one)
    NOTCHES = {
        'I': 'Q', 'II': 'E', 'III': 'V', 'IV': 'J', 'V': 'Z'
    }
    
    REFLECTOR = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'
    
    def __init__(self, rotors, positions):
        """
        rotors: list of 3 rotor names, e.g., ['III', 'II', 'I']
        positions: list of 3 starting positions (0-25)
        """
        self.rotors = [self.ROTORS[r] for r in rotors]
        self.rotor_names = rotors
        self.positions = positions.copy()
        self.initial_positions = positions.copy()
    
    def reset(self):
        """Reset rotor positions to initial state"""
        self.positions = self.initial_positions.copy()
    
    def set_positions(self, positions):
        """Set rotor positions to specific values"""
        self.positions = positions.copy()
    
    def get_positions(self):
        """Get current rotor positions"""
        return self.positions.copy()
    
    def step_rotors(self):
        """Advance rotors according to Enigma stepping rules"""
        # Check for double-stepping of middle rotor
        if self.rotor_at_notch(1):
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        elif self.rotor_at_notch(0):
            self.positions[1] = (self.positions[1] + 1) % 26
        
        # Always step the rightmost rotor
        self.positions[0] = (self.positions[0] + 1) % 26
    
    def rotor_at_notch(self, rotor_index):
        """Check if rotor is at its notch position"""
        notch = self.NOTCHES[self.rotor_names[rotor_index]]
        notch_pos = ord(notch) - ord('A')
        return self.positions[rotor_index] == notch_pos
    
    def encrypt_char(self, char):
        """Encrypt a single character"""
        if char not in string.ascii_uppercase:
            return char
        
        # Step rotors before encryption
        self.step_rotors()
        
        # Convert char to number (A=0, B=1, ...)
        pos = ord(char) - ord('A')
        
        # Forward through rotors (right to left)
        for i in range(3):
            pos = (pos + self.positions[i]) % 26
            pos = ord(self.rotors[i][pos]) - ord('A')
            pos = (pos - self.positions[i]) % 26
        
        # Through reflector
        pos = ord(self.REFLECTOR[pos]) - ord('A')
        
        # Backward through rotors (left to right)
        for i in range(2, -1, -1):
            pos = (pos + self.positions[i]) % 26
            pos = self.rotors[i].index(chr(pos + ord('A')))
            pos = (pos - self.positions[i]) % 26
        
        return chr(pos + ord('A'))
    
    def encrypt(self, text):
        """Encrypt a full message WITHOUT resetting positions"""
        result = []
        for char in text.upper():
            result.append(self.encrypt_char(char))
        return ''.join(result)

# ============================================================================
# FILE READING AND DATA PREPARATION
# ============================================================================
def read_text_from_file(filename):
    """
    Read text from a file with one letter per line
    Returns a string of uppercase letters only
    """
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Input file '{filename}' not found!")
    
    letters = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip().upper()
            # Only keep valid letters
            for char in line:
                if char in string.ascii_uppercase:
                    letters.append(char)
    
    text = ''.join(letters)
    print(f"Read {len(text)} letters from '{filename}'")
    return text

def generate_dataset_continuous(filename, rotor_order, initial_positions, 
                                num_samples=5000, message_length=100):
    """
    Generate training dataset by encrypting text from file with CONTINUOUS rotor advancement.
    
    Each sample captures the rotor positions at that point in the encryption sequence.
    The rotors advance naturally through the entire text, so different samples have
    different rotor positions.
    
    Args:
        filename: Path to input text file
        rotor_order: List of 3 rotor names (e.g., ['III', 'II', 'I'])
        initial_positions: List of 3 starting positions (e.g., [0, 0, 0] for AAA)
        num_samples: Number of training samples to generate
        message_length: Length of each training sample
    """
    # Read plaintext from file
    full_plaintext = read_text_from_file(filename)
    
    total_chars_needed = num_samples * message_length
    if len(full_plaintext) < total_chars_needed:
        print(f"WARNING: File has {len(full_plaintext)} letters but need {total_chars_needed}")
        print(f"Will repeat the text to generate enough samples")
        # Repeat text as needed
        repeats = (total_chars_needed // len(full_plaintext)) + 1
        full_plaintext = full_plaintext * repeats
    
    data = []
    print(f"\nGenerating {num_samples} training samples with continuous rotor advancement...")
    print(f"Rotor order: {rotor_order} (Left-Middle-Right)")
    print(f"Initial positions: {initial_positions} ({chr(initial_positions[0]+65)}{chr(initial_positions[1]+65)}{chr(initial_positions[2]+65)})")
    
    # Create enigma machine with starting configuration
    enigma = SimpleEnigma(rotor_order, initial_positions)
    
    # Encrypt the ENTIRE text in one continuous stream
    print(f"\nEncrypting {len(full_plaintext)} characters continuously...")
    full_ciphertext = enigma.encrypt(full_plaintext)
    print(f"✓ Encryption complete")
    
    # Now reset enigma to initial position to track positions during encryption
    enigma.set_positions(initial_positions)
    
    print(f"\nExtracting {num_samples} samples with rotor position tracking...")
    
    for i in range(num_samples):
        if (i + 1) % 1000 == 0:
            print(f"  Extracted {i + 1}/{num_samples} samples")
        
        # Calculate start index for this sample
        start_idx = i * message_length
        
        # Extract plaintext and ciphertext chunks
        plaintext = full_plaintext[start_idx:start_idx + message_length]
        ciphertext = full_ciphertext[start_idx:start_idx + message_length]
        
        # The rotor positions at the START of this message chunk
        # (before encrypting the first character)
        chunk_positions = enigma.get_positions()
        
        # Now advance the enigma through this chunk to be ready for next sample
        # (We don't use the output, just advancing the rotors)
        for char in plaintext:
            if char in string.ascii_uppercase:
                enigma.step_rotors()
        
        data.append({
            'ciphertext': ciphertext,
            'plaintext': plaintext,
            'rotors': rotor_order,
            'positions': chunk_positions  # Positions at START of this chunk
        })
    
    # Print statistics about position variation
    unique_positions = set()
    for item in data:
        pos_tuple = tuple(item['positions'])
        unique_positions.add(pos_tuple)
    
    print(f"\n✓ Dataset generated successfully")
    print(f"  Unique rotor position combinations: {len(unique_positions)}")
    print(f"  First sample positions: {data[0]['positions']} ({chr(data[0]['positions'][0]+65)}{chr(data[0]['positions'][1]+65)}{chr(data[0]['positions'][2]+65)})")
    print(f"  Last sample positions: {data[-1]['positions']} ({chr(data[-1]['positions'][0]+65)}{chr(data[-1]['positions'][1]+65)}{chr(data[-1]['positions'][2]+65)})")
    
    return data

# ============================================================================
# PYTORCH DATASET
# ============================================================================
class EnigmaDataset(Dataset):
    """PyTorch dataset for Enigma encrypted messages"""
    
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Convert ciphertext to one-hot encoded tensor
        ciphertext_encoded = self.encode_text(item['ciphertext'])
        
        # Convert positions to tensor (3 values, each 0-25)
        positions = torch.tensor(item['positions'], dtype=torch.long)
        
        return ciphertext_encoded, positions
    
    @staticmethod
    def encode_text(text):
        """Convert text to one-hot encoded tensor"""
        # Create tensor of shape (seq_len, 26)
        encoded = torch.zeros(len(text), 26)
        for i, char in enumerate(text):
            if char in string.ascii_uppercase:
                encoded[i, ord(char) - ord('A')] = 1
        return encoded

# ============================================================================
# NEURAL NETWORK MODEL
# ============================================================================
class EnigmaRotorClassifier(nn.Module):
    """
    CNN-based model to predict rotor positions from encrypted text
    
    OPTIMIZATION NOTES:
    - To change the number of convolutional layers, add/remove conv layers
    - To change the number of nodes in conv layers, modify the channel numbers
      (currently: 64 -> 128 -> 256)
    - To change the number of fully connected layers, add/remove fc layers
    - To change the number of nodes in FC layers, modify hidden_dim parameter
      (currently: 256)
    """
    
    def __init__(self, message_length=100, hidden_dim=256):
        """
        Args:
            message_length: Length of input message
            hidden_dim: Number of nodes in fully connected layers
                       *** CHANGE THIS TO ADJUST FC LAYER SIZE ***
        """
        super(EnigmaRotorClassifier, self).__init__()
        
        # ====================================================================
        # CONVOLUTIONAL LAYERS - MODIFY NUMBER OF LAYERS AND NODES HERE
        # ====================================================================
        # Current architecture: 3 conv layers with 64, 128, 256 channels
        # To add more layers: add self.conv4, self.batch_norm4, etc.
        # To change nodes: modify the channel numbers (e.g., 64 -> 128)
        
        self.conv1 = nn.Conv1d(26, 64, kernel_size=3, padding=1)      # *** 64 nodes ***
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)     # *** 128 nodes ***
        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)    # *** 256 nodes ***
        
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        
        # Calculate flattened size after convolutions
        # NOTE: If you change number of conv layers, adjust the divisor
        # Currently: 3 conv layers with pooling = divide by 2^3 = 8
        conv_output_size = message_length // 8 * 256  # 256 is last conv layer size
        
        # ====================================================================
        # FULLY CONNECTED LAYERS - MODIFY NUMBER OF LAYERS AND NODES HERE
        # ====================================================================
        # Current architecture: 2 FC layers with hidden_dim nodes each
        # To add more layers: add self.fc3, self.fc4, etc. and update forward()
        # To change nodes: modify hidden_dim parameter when creating model
        
        self.fc1 = nn.Linear(conv_output_size, hidden_dim)  # *** hidden_dim nodes ***
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)        # *** hidden_dim nodes ***
        
        # Three output heads (one for each rotor position, 26 classes each)
        self.rotor1_head = nn.Linear(hidden_dim, 26)
        self.rotor2_head = nn.Linear(hidden_dim, 26)
        self.rotor3_head = nn.Linear(hidden_dim, 26)
        
        self.relu = nn.ReLU()
        self.batch_norm1 = nn.BatchNorm1d(64)
        self.batch_norm2 = nn.BatchNorm1d(128)
        self.batch_norm3 = nn.BatchNorm1d(256)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len, 26)
        # Conv1d expects (batch_size, channels, seq_len)
        x = x.transpose(1, 2)
        
        # ====================================================================
        # FORWARD PASS THROUGH CONVOLUTIONAL LAYERS
        # If you add/remove conv layers, update this section
        # ====================================================================
        x = self.relu(self.batch_norm1(self.conv1(x)))
        x = self.pool(x)
        
        x = self.relu(self.batch_norm2(self.conv2(x)))
        x = self.pool(x)
        
        x = self.relu(self.batch_norm3(self.conv3(x)))
        x = self.pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        
        # ====================================================================
        # FORWARD PASS THROUGH FULLY CONNECTED LAYERS
        # If you add/remove FC layers, update this section
        # ====================================================================
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        
        # Output predictions for each rotor
        rotor1_out = self.rotor1_head(x)
        rotor2_out = self.rotor2_head(x)
        rotor3_out = self.rotor3_head(x)
        
        return rotor1_out, rotor2_out, rotor3_out

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_model(model, train_loader, val_loader, num_epochs=20, device='cuda'):
    """Train the Enigma rotor classifier"""
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)
    
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = [0, 0, 0]
        train_total = 0
        
        for batch_idx, (ciphertext, positions) in enumerate(train_loader):
            ciphertext = ciphertext.to(device)
            positions = positions.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            out1, out2, out3 = model(ciphertext)
            
            # Calculate loss for each rotor
            loss1 = criterion(out1, positions[:, 0])
            loss2 = criterion(out2, positions[:, 1])
            loss3 = criterion(out3, positions[:, 2])
            loss = loss1 + loss2 + loss3
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Calculate accuracy
            _, pred1 = torch.max(out1, 1)
            _, pred2 = torch.max(out2, 1)
            _, pred3 = torch.max(out3, 1)
            
            train_correct[0] += (pred1 == positions[:, 0]).sum().item()
            train_correct[1] += (pred2 == positions[:, 1]).sum().item()
            train_correct[2] += (pred3 == positions[:, 2]).sum().item()
            train_total += positions.size(0)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = [0, 0, 0]
        val_total = 0
        
        with torch.no_grad():
            for ciphertext, positions in val_loader:
                ciphertext = ciphertext.to(device)
                positions = positions.to(device)
                
                out1, out2, out3 = model(ciphertext)
                
                loss1 = criterion(out1, positions[:, 0])
                loss2 = criterion(out2, positions[:, 1])
                loss3 = criterion(out3, positions[:, 2])
                loss = loss1 + loss2 + loss3
                
                val_loss += loss.item()
                
                _, pred1 = torch.max(out1, 1)
                _, pred2 = torch.max(out2, 1)
                _, pred3 = torch.max(out3, 1)
                
                val_correct[0] += (pred1 == positions[:, 0]).sum().item()
                val_correct[1] += (pred2 == positions[:, 1]).sum().item()
                val_correct[2] += (pred3 == positions[:, 2]).sum().item()
                val_total += positions.size(0)
        
        # Calculate accuracies
        train_acc = [(c / train_total) * 100 for c in train_correct]
        val_acc = [(c / val_total) * 100 for c in val_correct]
        avg_val_acc = sum(val_acc) / 3
        
        scheduler.step(val_loss)
        
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print(f'Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'Train Acc - R1: {train_acc[0]:.2f}%, R2: {train_acc[1]:.2f}%, R3: {train_acc[2]:.2f}%')
        print(f'Val Loss: {val_loss/len(val_loader):.4f}')
        print(f'Val Acc - R1: {val_acc[0]:.2f}%, R2: {val_acc[1]:.2f}%, R3: {val_acc[2]:.2f}%')
        
        # Save best model
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            torch.save(model.state_dict(), 'best_enigma_model.pth')
            print(f'✓ Saved best model with avg accuracy: {avg_val_acc:.2f}%')
    
    return model

# ============================================================================
# PREDICTION AND TESTING
# ============================================================================
def predict_positions(model, ciphertext, device='cuda'):
    """Predict rotor positions from encrypted message"""
    model.eval()
    
    # Encode ciphertext
    encoded = EnigmaDataset.encode_text(ciphertext).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out1, out2, out3 = model(encoded)
        
        _, pred1 = torch.max(out1, 1)
        _, pred2 = torch.max(out2, 1)
        _, pred3 = torch.max(out3, 1)
        
        # Get confidence scores
        probs1 = torch.softmax(out1, dim=1)[0]
        probs2 = torch.softmax(out2, dim=1)[0]
        probs3 = torch.softmax(out3, dim=1)[0]
        
        confidence1 = probs1[pred1].item() * 100
        confidence2 = probs2[pred2].item() * 100
        confidence3 = probs3[pred3].item() * 100
    
    predictions = [pred1.item(), pred2.item(), pred3.item()]
    confidences = [confidence1, confidence2, confidence3]
    
    return predictions, confidences

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == '__main__':
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}\n')
    
    # Generate dataset with continuous rotor advancement
    print('=' * 60)
    print('GENERATING DATASET WITH CONTINUOUS ROTOR ADVANCEMENT')
    print('=' * 60)
    
    try:
        dataset = generate_dataset_continuous(
            filename=INPUT_TEXT_FILE,
            rotor_order=DEFAULT_ROTOR_ORDER,
            initial_positions=DEFAULT_POSITIONS,
            num_samples=NUM_TRAINING_SAMPLES,
            message_length=MESSAGE_LENGTH
        )
    except FileNotFoundError as e:
        print(f"\nERROR: Input file '{INPUT_TEXT_FILE}' not found!")
        print("Please create this file with one letter per line.")
        print("Example content:")
        print("A")
        print("B")
        print("C")
        print("...")
        raise FileNotFoundError(f"Required input file '{INPUT_TEXT_FILE}' not found. Please create it first.") from e
    
    # Split into train/val
    split_idx = int(0.8 * len(dataset))
    train_data = dataset[:split_idx]
    val_data = dataset[split_idx:]
    
    train_dataset = EnigmaDataset(train_data)
    val_dataset = EnigmaDataset(val_data)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    print(f'\nTraining samples: {len(train_data)}')
    print(f'Validation samples: {len(val_data)}')
    
    # Create model
    print('\n' + '=' * 60)
    print('CREATING MODEL')
    print('=' * 60)
    
    # ========================================================================
    # MODEL CREATION - CHANGE hidden_dim TO ADJUST FC LAYER SIZE
    # ========================================================================
    model = EnigmaRotorClassifier(
        message_length=MESSAGE_LENGTH,
        hidden_dim=256  # *** CHANGE THIS TO MODIFY FC LAYER SIZE ***
    )
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Total parameters: {total_params:,}')
    
    # Train model
    print('\n' + '=' * 60)
    print('TRAINING MODEL')
    print('=' * 60)
    model = train_model(model, train_loader, val_loader, num_epochs=20, device=device)
    
    # Test predictions on multiple samples with different positions
    print('\n' + '=' * 60)
    print('TESTING PREDICTIONS ON VARIED POSITIONS')
    print('=' * 60)
    
    # Test on first, middle, and last samples (different rotor positions)
    test_indices = [0, len(dataset)//2, len(dataset)-1]
    
    for idx in test_indices:
        test_sample = dataset[idx]
        
        print(f'\n--- Test Sample {idx} ---')
        print(f'True positions: {test_sample["positions"]} ({chr(test_sample["positions"][0]+65)}{chr(test_sample["positions"][1]+65)}{chr(test_sample["positions"][2]+65)})')
        print(f'Ciphertext: {test_sample["ciphertext"][:50]}...')
        
        predictions, confidences = predict_positions(model, test_sample['ciphertext'], device)
        
        print(f'\nPredictions:')
        print(f'  Rotor 1: {predictions[0]} ({chr(predictions[0]+65)}) - Confidence: {confidences[0]:.1f}%')
        print(f'  Rotor 2: {predictions[1]} ({chr(predictions[1]+65)}) - Confidence: {confidences[1]:.1f}%')
        print(f'  Rotor 3: {predictions[2]} ({chr(predictions[2]+65)}) - Confidence: {confidences[2]:.1f}%')
        
        print(f'\nAccuracy:')
        print(f'  Rotor 1: {"✓ CORRECT" if predictions[0] == test_sample["positions"][0] else "✗ INCORRECT"}')
        print(f'  Rotor 2: {"✓ CORRECT" if predictions[1] == test_sample["positions"][1] else "✗ INCORRECT"}')
        print(f'  Rotor 3: {"✓ CORRECT" if predictions[2] == test_sample["positions"][2] else "✗ INCORRECT"}')
    
    print('\n' + '=' * 60)
    print('TRAINING COMPLETE')
    print('=' * 60)
    print('Model saved as: best_enigma_model.pth')

Using device: cpu

GENERATING DATASET WITH CONTINUOUS ROTOR ADVANCEMENT
Read 37231 letters from 'C:\Users\acool\1 Capstone\bee movie script processed.txt'
Will repeat the text to generate enough samples

Generating 5000 training samples with continuous rotor advancement...
Rotor order: ['III', 'II', 'I'] (Left-Middle-Right)
Initial positions: [0, 0, 0] (AAA)

Encrypting 521234 characters continuously...
✓ Encryption complete

Extracting 5000 samples with rotor position tracking...
  Extracted 1000/5000 samples
  Extracted 2000/5000 samples
  Extracted 3000/5000 samples
  Extracted 4000/5000 samples
  Extracted 5000/5000 samples

✓ Dataset generated successfully
  Unique rotor position combinations: 169
  First sample positions: [0, 0, 0] (AAA)
  Last sample positions: [24, 2, 15] (YCP)

Training samples: 4000
Validation samples: 1000

CREATING MODEL
Total parameters: 1,001,742

TRAINING MODEL

Epoch 1/20
Train Loss: 8.6171
Train Acc - R1: 7.45%, R2: 8.12%, R3: 4.05%
Val Loss: 8.4351
Va